# Notebook 3 — Extraction et nettoyage des données brutes
## Station de Kara — Production des fichiers CSV propres

---

## Contexte et objectif

Ce notebook est le **premier maillon du pipeline de production** dans le projet IDF de Kara.
Après l'exploration visuelle des données brutes (Notebooks 1 et 2), on passe ici à
l'**extraction structurée** et au **nettoyage systématique** des deux sources de données :

| Source | Fichier brut | Description | Période |
|--------|-------------|-------------|---------|
| **Pluviomètre** | `KARA PLUIE JOURNALIERE.xls` | Pluies journalières en mm | 1980–2014 |
| **Pluviographe** | `INTENSITES DE PLUIE -KARA.xlsx` | Intensités par événement | 2004–2014 |

### Pourquoi ce notebook est-il nécessaire ?

Les fichiers bruts présentent des défis spécifiques :
- **Pluviomètre** : format bloc (année × mois × jours 1–31), deux formats coexistent
  dans le même fichier (ancien 1980–2009 vs nouveau 2010–2014), zéro encodé comme `.`
- **Pluviographe** : fichiers Excel multi-feuilles avec structure en blocs mensuels.
  Les durées enregistrées sont des **durées brutes d'événements**, pas des durées
  standardisées. Une étape d'agrégation aux durées standards est nécessaire.

### Protocole de nettoyage — Pluviomètre

| Étape | Règle | Justification |
|-------|-------|---------------|
| Valeurs négatives | → NaN | Physiquement impossible |
| Doublons | Supprimés | Erreurs de saisie |
| Erreurs de saisie systématiques | Supprimées | Valeurs > 200 mm en saison sèche (nov–mars) : impossibles à Kara, motif de date répété chaque année |
| Outliers extrêmes (saison pluies) | IQR × 3 → flag (conservés) | Extrêmes réels possibles en saison des pluies |
| Valeurs manquantes | Imputation médiane mensuelle si taux < 15% | Médiane robuste aux extrêmes |

### Protocole de nettoyage — Pluviographe

| Étape | Règle | Justification |
|-------|-------|---------------|
| Doublons | Supprimés | Erreurs de fusion |
| Durées non-standards | Agrégation aux 8 durées standards | Les courbes IDF exigent des durées homogènes |
| Outliers | IQR × 3 par durée (conservés) | Extrêmes réels possibles |

### Livrables

| Fichier | Contenu |
|---------|---------|
| `data/pluviometre_clean.csv` | `[date, pluie_mm, flag_outlier, flag_manquant]` |
| `data/pluviographe_clean.csv` | `[date, duree_min, intensite_mm_h, flag_outlier]` (durées standardisées) |

---
**Auteur :** AY2K
**Encadreur :** Prof. Titembaye Donald
**Établissement :** École Polytechnique de Lomé (EPL)
**Projet :** Modélisation des courbes IDF — Station de Kara, Nord Togo


## 0. Configuration et imports

On initialise les répertoires et les bibliothèques nécessaires.

In [ ]:
import os
import warnings
import datetime
from pathlib import Path

# On filtre uniquement les avertissements de convergence scipy
# pour ne pas masquer les erreurs réelles
warnings.filterwarnings('ignore', category=RuntimeWarning, module='scipy')
warnings.filterwarnings('ignore', category=FutureWarning)

NB      = 'n3'
FIG_DIR = f'figures/{NB}'
DATA_DIR = 'Donnees'

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs('data', exist_ok=True)

print(f'Répertoire figures : {FIG_DIR}')
print(f'Répertoire données : {DATA_DIR}')
print(f'Répertoire CSV     : data/')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.6,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'figure.dpi':       100,
})

print('Imports chargés avec succès')

---
## Section 1 — Extraction du pluviomètre (données journalières)

### Structure du fichier brut

Le fichier `KARA PLUIE JOURNALIERE.xls` contient **35 blocs annuels empilés** (1980–2014).
Chaque bloc a la structure suivante :

```
Ligne d'en-tête :   JOUR | JAN | FEV | MARS | ... | DEC
Lignes de données : 1–31 (chaque ligne = un jour du mois)
```

### Défi : deux formats coexistent

| Période | Format | Zéro encodé comme | Col. JOUR |
|---------|--------|-------------------|-----------|
| 1980–2009 | Ancien (export imprimante) | `.` (point) | Colonne 1 |
| 2010–2014 | Nouveau (tableur) | `NaN` | Colonne 2 |


In [ ]:
# ==============================================================================
# SECTION 1.1 — Résolution robuste du chemin du fichier
# ==============================================================================
f_pluvio = next((p for p in [
    Path('Donnees/KARA PLUIE JOURNALIERE.xls'),
    Path('Donnees/KARA PLUIE JOURNALIERE.xlsx'),
    Path('Projet_stage/Donnees/KARA PLUIE JOURNALIERE.xls'),
] if p.exists()), None)

if f_pluvio is None:
    raise FileNotFoundError(
        "Fichier pluviomètre introuvable. "
        "Vérifiez que 'KARA PLUIE JOURNALIERE.xls' est dans le dossier Donnees/"
    )

print(f'Fichier trouvé : {f_pluvio}')

In [ ]:
# ==============================================================================
# SECTION 1.2 — Carte structurelle du fichier brut
# ==============================================================================
# Positions de début de chaque bloc annuel, déterminées par inspection (n1).

ANNEE_MARKERS = {
    1980: 13, 1981: 80,  1982: 147, 1983: 214, 1984: 281, 1985: 348,
    1986: 415, 1987: 482, 1988: 549, 1989: 616, 1990: 683, 1991: 750,
    1992: 817, 1993: 884, 1994: 951, 1995: 1018, 1996: 1085, 1997: 1152,
    1998: 1219, 1999: 1286, 2000: 1345, 2001: 1395, 2002: 1445, 2003: 1497,
    2004: 1546, 2005: 1598, 2006: 1648, 2007: 1698, 2008: 1744, 2009: 1790,
    2010: 1836, 2011: 1880, 2012: 1925, 2013: 1972, 2014: 2013,
}

df_brut = pd.read_excel(f_pluvio, engine='xlrd', sheet_name=0, header=None)

print(f'Dimensions du fichier brut : {df_brut.shape[0]} lignes × {df_brut.shape[1]} colonnes')
print(f"Nombre d'années détectées  : {len(ANNEE_MARKERS)}")
print(f'Période couverte           : {min(ANNEE_MARKERS)} – {max(ANNEE_MARKERS)}')

In [ ]:
# ==============================================================================
# SECTION 1.3 — Extraction des données journalières
# ==============================================================================
records    = []
n_annees_ok = 0

for annee, ligne_debut in ANNEE_MARKERS.items():
    try:
        ancien_format  = (annee <= 2009)
        col_mois_debut = 2 if ancien_format else 3
        ligne_mois     = ligne_debut + 2

        for jour in range(1, 32):
            ligne_data = ligne_mois + 2 + jour
            if ligne_data >= len(df_brut):
                continue
            for mois_idx in range(12):
                col_idx = col_mois_debut + mois_idx
                if col_idx >= df_brut.shape[1]:
                    continue
                val   = df_brut.iloc[ligne_data, col_idx]
                pluie = None
                if isinstance(val, str):
                    val_clean = val.strip()
                    if val_clean in ('.', ''):
                        pluie = 0.0
                    else:
                        try:
                            pluie = float(val_clean)
                        except ValueError:
                            pass
                elif isinstance(val, (int, float)):
                    pluie = float(val)
                if pluie is not None and not np.isnan(pluie):
                    try:
                        date_val = pd.Timestamp(datetime.date(annee, mois_idx + 1, jour))
                        records.append({'date': date_val, 'pluie_mm': pluie})
                    except (ValueError, OverflowError):
                        pass
        n_annees_ok += 1
    except Exception as e:
        print(f'  Erreur année {annee} : {e}')

df_pm = pd.DataFrame(records).sort_values('date').reset_index(drop=True)

print(f"\n=== Résultat de l'extraction pluviomètre ===")
print(f"Années traitées      : {n_annees_ok}/{len(ANNEE_MARKERS)}")
print(f"Total jours extraits : {len(df_pm)}")
print(f"Période              : {df_pm.date.min().date()} → {df_pm.date.max().date()}")
print(f"Pluie maximale brute : {df_pm.pluie_mm.max():.1f} mm")
df_pm.head(6)

---
## Section 2 — Nettoyage du pluviomètre

### Protocole de nettoyage en 5 étapes

#### Étape 1 : Valeurs négatives → NaN
Impossibles physiquement (erreurs de saisie).

#### Étape 2 : Suppression des doublons
Deux enregistrements pour la même date indiquent une erreur de fusion.

#### Étape 3 (NOUVEAU) : Suppression des erreurs de saisie systématiques
L'inspection des données révèle **20 valeurs aberrantes extrêmes** (1 000–1 766 mm)
toujours datées du **2 février** de chaque année 1980–1999.

Ces valeurs sont des **erreurs de saisie manifestes**, et non des extrêmes hydrologiques :
- Elles surviennent en **saison sèche** (février à Kara : précipitations quasi nulles)
- Elles ont une valeur absolue **physiquement impossible** (record mondial ≈ 1 825 mm/jour)
- Elles présentent un **motif systématique** : même jour, même mois, chaque année

**Règle appliquée :** toute valeur > 200 mm/jour en saison sèche (novembre–mars)
est considérée comme erreur de saisie et mise à NaN.

Le seuil de 200 mm est conservateur : à Kara, les pluies journalières dépassent
rarement 150 mm même en saison des pluies.

#### Étape 4 : Détection des valeurs aberrantes (IQR × 3) sur la saison des pluies
Le facteur 3 (au lieu du 1,5 classique) est justifié : les averses convectives
d'Afrique de l'Ouest peuvent atteindre 100–180 mm/jour sans être des erreurs.
Ces outliers sont **flaggés mais conservés** (ils peuvent être de vrais extrêmes).

#### Étape 5 : Imputation des valeurs manquantes
Si le taux de manquants est < 15 % : imputation par la médiane mensuelle.


In [ ]:
# ==============================================================================
# SECTION 2 — Nettoyage du pluviomètre
# ==============================================================================
df_pm_clean = df_pm.copy()
n_initial   = len(df_pm_clean)

# --- Étape 1 : Valeurs négatives → NaN ---
mask_neg = df_pm_clean['pluie_mm'] < 0
n_neg    = mask_neg.sum()
df_pm_clean.loc[mask_neg, 'pluie_mm'] = np.nan
print(f'Étape 1 — Valeurs négatives → NaN          : {n_neg}')

# --- Étape 2 : Suppression des doublons ---
n_dup = df_pm_clean.duplicated(subset=['date']).sum()
df_pm_clean = df_pm_clean.drop_duplicates(subset=['date'])
print(f'Étape 2 — Doublons supprimés                : {n_dup}')

# --- Étape 3 : Suppression des erreurs de saisie systématiques ---
# Critère : pluie > 200 mm en saison sèche (novembre à mars inclus)
SEUIL_SAISON_SECHE  = 200.0          # mm — au-dessus, impossible en saison sèche
MOIS_SAISON_SECHE   = [11, 12, 1, 2, 3]

df_pm_clean['mois_tmp'] = df_pm_clean['date'].dt.month
mask_saison_seche   = df_pm_clean['mois_tmp'].isin(MOIS_SAISON_SECHE)
mask_impossible     = mask_saison_seche & (df_pm_clean['pluie_mm'] > SEUIL_SAISON_SECHE)
n_impossible        = mask_impossible.sum()

df_pm_clean.loc[mask_impossible, 'pluie_mm'] = np.nan
df_pm_clean = df_pm_clean.drop(columns=['mois_tmp'])

print(f'Étape 3 — Erreurs de saisie saison sèche    : {n_impossible} valeurs supprimées')
print(f'          (seuil : > {SEUIL_SAISON_SECHE:.0f} mm en nov–mars)')

# --- Étape 4 : Détection des outliers IQR × 3 (saison des pluies) ---
# On calcule l'IQR uniquement sur les valeurs non-nulles de saison des pluies
pluies_saison = df_pm_clean.loc[
    df_pm_clean['pluie_mm'] > 0, 'pluie_mm'
].dropna()
q1, q3      = pluies_saison.quantile([0.25, 0.75])
iqr         = q3 - q1
seuil_upper = q3 + 3 * iqr

df_pm_clean['flag_outlier'] = df_pm_clean['pluie_mm'] > seuil_upper
n_outliers   = df_pm_clean['flag_outlier'].sum()
print(f'Étape 4 — Outliers flaggés (IQR×3, seuil={seuil_upper:.1f} mm) : {n_outliers}')

# --- Étape 5 : Valeurs manquantes ---
df_pm_clean['flag_manquant'] = df_pm_clean['pluie_mm'].isna()
n_manquants = df_pm_clean['flag_manquant'].sum()
pct_manq    = 100 * n_manquants / len(df_pm_clean)
print(f'Étape 5 — Valeurs manquantes                : {n_manquants} ({pct_manq:.1f}%)')

if pct_manq < 15:
    df_pm_clean['mois'] = df_pm_clean['date'].dt.month
    medianes = df_pm_clean.groupby('mois')['pluie_mm'].transform('median')
    df_pm_clean.loc[df_pm_clean['flag_manquant'], 'pluie_mm'] =         medianes[df_pm_clean['flag_manquant']]
    df_pm_clean = df_pm_clean.drop(columns=['mois'])
    print('           → Imputation par médiane mensuelle appliquée')
else:
    print('           → Taux ≥ 15% : pas d\'imputation automatique')

print(f'\n=== Résumé pluviomètre ===')
print(f'Lignes           : {n_initial} → {len(df_pm_clean)}')
print(f'Pluie max finale : {df_pm_clean.pluie_mm.max():.1f} mm  (attendu : < 200 mm en saison sèche)')
df_pm_clean.describe()

---
## Section 3 — Extraction et nettoyage du pluviographe

### Structure du fichier pluviographe

Le fichier `INTENSITES DE PLUIE -KARA.xlsx` contient **11 feuilles** (2004–2014).
Chaque feuille couvre les 5 mois de la saison des pluies (juin–octobre),
avec 2 événements par jour (matin et soir), chacun décrit par :
- hauteur (mm) et durée brute de l'événement (min)

### Problème des durées non-standardisées

Le pluviographe enregistre les **durées brutes** des événements pluvieux (mesurées
au pluviographe à augets basculeurs). Ces durées sont continues et non-standardisées :
on obtient 86 valeurs uniques différentes, dont beaucoup ne comptent que 1 à 4
événements sur 11 ans.

Pour construire des courbes IDF, on a besoin de **maxima annuels PAR DURÉE FIXE**.
On procède à une **agrégation** : pour chaque durée standard D, on calcule l'intensité
maximale sur D minutes à partir de la lame d'eau de l'événement.

### Durées standardisées retenues

| Durée (min) | Durée (h) | Usage IDF |
|-------------|-----------|-----------|
| 30 | 0h30 | Réseaux d'assainissement |
| 60 | 1h   | Standard hydrologique |
| 90 | 1h30 | Bassins versants urbains |
| 120 | 2h  | Ouvrages de rétention |
| 180 | 3h  | Petits barrages |
| 240 | 4h  | Barrages |
| 360 | 6h  | Grands ouvrages |
| 720 | 12h | Crues de durée longue |

### Méthode d'agrégation

Pour un événement de hauteur H mm et de durée brute D_brut min :
- Si D_brut ≥ D_std : l'événement peut contribuer à la durée D_std.
  L'intensité sur D_std est estimée par I = (H / D_brut) × 60 mm/h
  (on suppose une intensité constante sur la durée de l'événement).
- La durée standard est retenue si D_std est dans l'intervalle [0.5×D_brut, 2×D_brut]
  pour limiter l'extrapolation.

Cette approche est conservative et documentée. Les durées très courtes (< 30 min)
ont été exclues car le pluviographe à augets ne garantit pas une précision suffisante
à ces échelles de temps.


In [ ]:
# ==============================================================================
# SECTION 3.1 — Résolution du chemin du pluviographe
# ==============================================================================
f_pg = next((p for p in [
    Path('Donnees/INTENSITES DE PLUIE -KARA.xlsx'),
    Path('Donnees/INTENSITES DE PLUIE-KARA.xlsx'),
    Path('Projet_stage/Donnees/INTENSITES DE PLUIE -KARA.xlsx'),
] if p.exists()), None)

if f_pg is None:
    raise FileNotFoundError('Fichier pluviographe introuvable')

print(f'Fichier trouvé : {f_pg}')

In [ ]:
# ==============================================================================
# SECTION 3.2 — Extraction des événements bruts
# ==============================================================================
BLOCS_MOIS = {
    'Juin':      {'jour_col': 1,  'hm_col': 2,  'dm_col': 3,  'hs_col': 4,  'ds_col': 5},
    'Juillet':   {'jour_col': 7,  'hm_col': 8,  'dm_col': 9,  'hs_col': 10, 'ds_col': 11},
    'Août':      {'jour_col': 13, 'hm_col': 14, 'dm_col': 15, 'hs_col': 16, 'ds_col': 17},
    'Septembre': {'jour_col': 19, 'hm_col': 20, 'dm_col': 21, 'hs_col': 22, 'ds_col': 23},
    'Octobre':   {'jour_col': 25, 'hm_col': 26, 'dm_col': 27, 'hs_col': 28, 'ds_col': 29},
}
MOIS_NUM = {'Juin': 6, 'Juillet': 7, 'Août': 8, 'Septembre': 9, 'Octobre': 10}

# Durées standardisées retenues pour les courbes IDF
DUREES_STANDARDS = [30, 60, 90, 120, 180, 240, 360, 720]  # minutes


def extraire_annee_pluviographe(fichier, annee):
    """Extrait tous les événements bruts d'une feuille annuelle."""
    df_yr = pd.read_excel(fichier, sheet_name=str(annee), header=None)
    data  = df_yr.iloc[6:37].reset_index(drop=True)
    records = []
    for mois_nom, cols in BLOCS_MOIS.items():
        for periode, h_col, d_col in [
            ('matin',  cols['hm_col'], cols['dm_col']),
            ('soir',   cols['hs_col'], cols['ds_col']),
        ]:
            for day_idx in range(31):
                h = data.iloc[day_idx, h_col]
                d = data.iloc[day_idx, d_col]
                if pd.notna(h) and pd.notna(d):
                    try:
                        h_val = float(h)
                        d_val = float(d)
                        if h_val > 0 and d_val > 0:
                            try:
                                date_val = pd.Timestamp(
                                    datetime.date(annee, MOIS_NUM[mois_nom], day_idx + 1))
                                records.append({
                                    'date':        date_val,
                                    'annee':       annee,
                                    'mois':        MOIS_NUM[mois_nom],
                                    'periode':     periode,
                                    'hauteur_mm':  h_val,
                                    'duree_brute_min': d_val,
                                    # Intensité brute de l'événement
                                    'intensite_brute_mm_h': h_val / (d_val / 60),
                                })
                            except (ValueError, OverflowError):
                                pass
                    except (ValueError, ZeroDivisionError):
                        pass
    return pd.DataFrame(records)


frames = [extraire_annee_pluviographe(f_pg, yr) for yr in range(2004, 2015)]
df_pg_brut = pd.concat(frames, ignore_index=True)

print(f"=== Résultat extraction brute ===")
print(f"Total événements : {len(df_pg_brut):,}")
print(f"Années           : {df_pg_brut.annee.min()} – {df_pg_brut.annee.max()}")
print(f"Hauteur max      : {df_pg_brut.hauteur_mm.max():.1f} mm")
print(f"Durée max (brute): {df_pg_brut.duree_brute_min.max():.0f} min")
df_pg_brut.head(5)

In [ ]:
# ==============================================================================
# SECTION 3.3 — Agrégation aux durées standardisées
# ==============================================================================
# Pour chaque événement brut (hauteur H, durée D_brut), on estime l'intensité
# sur chaque durée standard D_std selon la règle :
#   - L'événement contribue à D_std si D_std est "compatible" avec D_brut
#   - Compatibilité : D_brut ∈ [D_std × 0.5 ; D_std × 2.0]
#     (on évite d'extrapoler à plus du double ou moins de la moitié)
#   - Intensité estimée : I(D_std) = (H / D_brut) × 60 mm/h
#     (hypothèse : intensité uniforme sur la durée de l'événement)

records_std = []

for _, evt in df_pg_brut.iterrows():
    H      = evt['hauteur_mm']
    D_brut = evt['duree_brute_min']
    I_brut = evt['intensite_brute_mm_h']   # mm/h

    for D_std in DUREES_STANDARDS:
        # Critère de compatibilité
        if D_brut >= D_std * 0.5 and D_brut <= D_std * 2.0:
            # Intensité sur la durée standard
            I_std = I_brut  # même intensité, car on suppose uniformité
            records_std.append({
                'date':          evt['date'],
                'annee':         evt['annee'],
                'mois':          evt['mois'],
                'duree_min':     D_std,
                'intensite_mm_h': I_std,
            })

df_pg = pd.DataFrame(records_std)

# Supprimer les doublons (même date + même durée standard → garder le max)
df_pg = (df_pg
         .groupby(['date', 'annee', 'mois', 'duree_min'], as_index=False)
         ['intensite_mm_h'].max())

print(f"=== Pluviographe — durées standardisées ===")
print(f"Nombre d'enregistrements : {len(df_pg):,}")
print(f"\nNombre d'événements par durée standard :")
for d in DUREES_STANDARDS:
    subset = df_pg[df_pg.duree_min == d]
    n_annees = subset['annee'].nunique()
    print(f"  {d:>4} min : {len(subset):>4} événements | {n_annees} années")

In [ ]:
# ==============================================================================
# SECTION 3.4 — Nettoyage du pluviographe standardisé
# ==============================================================================
df_pg_clean = df_pg.copy()

# Détection des outliers par durée (IQR × 3)
# Chaque durée a sa propre distribution d'intensités → seuil par durée
df_pg_clean['flag_outlier'] = False
n_outliers_pg = 0

for duree, grp in df_pg_clean.groupby('duree_min'):
    q1, q3 = grp['intensite_mm_h'].quantile([0.25, 0.75])
    iqr    = q3 - q1
    seuil  = q3 + 3 * iqr
    mask   = (df_pg_clean['duree_min'] == duree) & (df_pg_clean['intensite_mm_h'] > seuil)
    df_pg_clean.loc[mask, 'flag_outlier'] = True
    n_outliers_pg += mask.sum()

print(f"Outliers flaggés (IQR×3 par durée) : {n_outliers_pg}")
print(f"\nRésumé par durée :")
for d in DUREES_STANDARDS:
    s   = df_pg_clean[df_pg_clean.duree_min == d]
    n_ok = (~s['flag_outlier']).sum()
    n_fl = s['flag_outlier'].sum()
    if len(s) > 0:
        print(f"  {d:>4} min : {n_ok} événements OK | {n_fl} flaggés")

---
## Section 4 — Visualisations de contrôle


In [ ]:
# ==============================================================================
# FIGURE 1 — Pluviomètre : impact du nettoyage (avant / après)
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avant : distribution brute avec valeurs corrompues
vals_avant = df_pm['pluie_mm'].dropna()
vals_avant_clip = vals_avant[vals_avant <= 250]   # On tronque l'affichage
axes[0].hist(vals_avant_clip, bins=50, color='#EF9A9A', edgecolor='white')
axes[0].set_title('Avant nettoyage (tronqué à 250 mm)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Pluie journalière (mm)')
axes[0].set_ylabel('Fréquence')
axes[0].text(0.97, 0.95, f'{(vals_avant > 250).sum()} valeurs > 250 mm\n(erreurs de saisie, non affichées)',
             transform=axes[0].transAxes, ha='right', va='top', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='#FFCDD2', alpha=0.8))

# Après : distribution nettoyée
vals_apres  = df_pm_clean.loc[~df_pm_clean['flag_outlier'], 'pluie_mm'].dropna()
vals_out    = df_pm_clean.loc[ df_pm_clean['flag_outlier'], 'pluie_mm'].dropna()
axes[1].hist(vals_apres, bins=50, color='#81C784', edgecolor='white', label='Normal')
if len(vals_out) > 0:
    axes[1].hist(vals_out, bins=15, color='#E53935', edgecolor='white',
                 alpha=0.7, label=f'Outlier flaggé ({len(vals_out)})')
axes[1].set_title(f'Après nettoyage (max={df_pm_clean.pluie_mm.max():.1f} mm)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Pluie journalière (mm)')
axes[1].legend()

plt.suptitle('Pluviomètre — Avant/Après nettoyage', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig1_avant_apres_pluviometre.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig1 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 2 — Pluviographe : nombre d'événements par durée et par année
# ==============================================================================
fig, ax = plt.subplots(figsize=(13, 5))

annees  = sorted(df_pg_clean['annee'].unique())
couleurs = plt.cm.tab10(np.linspace(0, 1, len(DUREES_STANDARDS)))

x = np.arange(len(DUREES_STANDARDS))
width = 0.06
offsets = np.linspace(-(len(annees) - 1) * width / 2,
                       (len(annees) - 1) * width / 2, len(annees))

for j, annee in enumerate(annees):
    counts = [len(df_pg_clean[(df_pg_clean.duree_min == d) & (df_pg_clean.annee == annee)])
              for d in DUREES_STANDARDS]
    ax.bar(x + offsets[j], counts, width=width * 0.85, label=str(annee),
           alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([f'{d}\nmin' for d in DUREES_STANDARDS])
ax.set_ylabel("Nombre d'événements")
ax.set_title('Pluviographe standardisé — Événements par durée et par année',
             fontsize=13, fontweight='bold')
ax.legend(title='Année', ncol=4, fontsize=8)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig2_evenements_duree_annee.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig2 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 3 — Complétude par année (pluviomètre)
# ==============================================================================
df_pm_clean['annee'] = df_pm_clean['date'].dt.year
completude_pm = df_pm_clean.groupby('annee').apply(
    lambda g: 100 * g['pluie_mm'].notna().sum() / 365
).reset_index(name='completude_pct')

fig, ax = plt.subplots(figsize=(13, 4))
bars = ax.bar(completude_pm['annee'], completude_pm['completude_pct'],
              color='#42A5F5', edgecolor='white')
ax.axhline(80, color='orange', linestyle='--', linewidth=1.5, label='Seuil 80%')
for bar, val in zip(bars, completude_pm['completude_pct']):
    if val < 80:
        bar.set_color('#EF9A9A')
ax.set_ylabel('Complétude (%)')
ax.set_xlabel('Année')
ax.set_title('Pluviomètre — Taux de complétude par année', fontweight='bold')
ax.legend()
ax.set_ylim(0, 110)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig3_completude.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig3 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 4 — Intensités observées par durée standard (boîtes à moustaches)
# ==============================================================================
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(13, 5))

data_bp = [df_pg_clean.loc[
    (df_pg_clean['duree_min'] == d) & (~df_pg_clean['flag_outlier']),
    'intensite_mm_h'].values for d in DUREES_STANDARDS]

bp = ax.boxplot(data_bp, patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))
couleurs_bp = plt.cm.Blues(np.linspace(0.4, 0.9, len(DUREES_STANDARDS)))
for patch, col in zip(bp['boxes'], couleurs_bp):
    patch.set_facecolor(col)

ax.set_xticks(range(1, len(DUREES_STANDARDS) + 1))
ax.set_xticklabels([f'{d} min' for d in DUREES_STANDARDS])
ax.set_xlabel('Durée standard')
ax.set_ylabel('Intensité (mm/h)')
ax.set_title('Pluviographe — Distribution des intensités par durée standardisée',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig4_boxplots_intensites.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig4 sauvegardée')

---
## Section 5 — Export des fichiers CSV nettoyés


In [ ]:
# --- Export pluviomètre ---
out_pm = df_pm_clean[['date', 'pluie_mm', 'flag_outlier', 'flag_manquant']].copy()
out_pm.to_csv('data/pluviometre_clean.csv', index=False)
print(f'data/pluviometre_clean.csv : {len(out_pm)} lignes')
print(f'  Pluie max : {out_pm.pluie_mm.max():.1f} mm')
print(f'  Flag outlier : {out_pm.flag_outlier.sum()} valeurs')
print(f'  Flag manquant : {out_pm.flag_manquant.sum()} valeurs')

# --- Export pluviographe ---
out_pg = df_pg_clean[['date', 'duree_min', 'intensite_mm_h', 'flag_outlier']].copy()
out_pg.to_csv('data/pluviographe_clean.csv', index=False)
print(f'\ndata/pluviographe_clean.csv : {len(out_pg)} lignes')
print(f'  Durées : {sorted(out_pg.duree_min.unique())}')
print(f'  Intensité max : {out_pg.intensite_mm_h.max():.1f} mm/h')

---
## Rapport de nettoyage

| Décision | Pluviomètre | Pluviographe |
|----------|-------------|--------------|
| Valeurs négatives | → NaN | Non applicables |
| Doublons | Supprimés (sur `date`) | Supprimés (max gardé par date+durée) |
| Erreurs de saisie systématiques | **Supprimées** (> 200 mm en saison sèche) | N/A |
| Outliers | IQR × 3 → flag (conservés) | IQR × 3 par durée → flag (conservés) |
| Manquants | Imputation médiane mensuelle si < 15% | Non imputés |
| Durées | N/A | **Standardisées** : 30, 60, 90, 120, 180, 240, 360, 720 min |
| **Livrable** | `data/pluviometre_clean.csv` | `data/pluviographe_clean.csv` |

**Correction critique appliquée :** les 20 valeurs pluviométriques aberrantes en saison
sèche (1 000–1 766 mm en février, chaque année 1980–1999) ont été identifiées comme
erreurs de saisie systématiques et supprimées. Sans cette correction, elles auraient
contaminé la série reconstituée 1980–2003 et invalidé les courbes IDF.

---
*Fin du Notebook 3*
